In [7]:
%%capture
!pip install genomic-benchmarks transformers scikit-learn torch

In [8]:
import random
import numpy as np
import torch
from genomic_benchmarks.dataset_getters.pytorch_datasets import HumanNontataPromoters
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Set seeds for reproducibility
SEED = 1908
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [9]:
# Load dataset
train_dataset = HumanNontataPromoters(split='train')
test_dataset = HumanNontataPromoters(split='test')

# Load Model and Tokenizer
model_name = 'InstaDeepAI/nucleotide-transformer-v2-50m-multi-species'
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(device)
model.eval()

AttributeError: 'EsmConfig' object has no attribute 'rope_theta'

In [ ]:
def get_embeddings(dataset, num_samples):
    # Limit samples for testing
    indices = list(range(min(num_samples, len(dataset))))

    embeddings_list = []
    labels_list = []
    batch_size = 8

    with torch.no_grad():
        for i in range(0, len(indices), batch_size):
            batch_indices = indices[i:i+batch_size]
            batch_data = [dataset[idx] for idx in batch_indices]
            sequences = [item[0] for item in batch_data]
            labels = [item[1] for item in batch_data]

            inputs = tokenizer(sequences, return_tensors='pt', padding=True, truncation=True).to(device)
            outputs = model(**inputs)

            # Mean pooling over tokens (dimension 1)
            embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            embeddings_list.append(embeddings)
            labels_list.extend(labels)

    return np.vstack(embeddings_list), np.array(labels_list)

# Test on subset: 1000 training, 500 test
print("Extracting embeddings for subset...")
X_train, y_train = get_embeddings(train_dataset, 1000)
X_test, y_test = get_embeddings(test_dataset, 500)

# Code to scale to full dataset (commented out):
# X_train, y_train = get_embeddings(train_dataset, len(train_dataset))
# X_test, y_test = get_embeddings(test_dataset, len(test_dataset))

In [ ]:
clf = LogisticRegression(max_iter=1000, random_state=SEED)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred):.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

**Genomic Language Model + Klassifikation**